In [1]:
# %% [markdown]
# # Diagnostic 1: Single-Batch Overfitting Test
# **Objective:** Verify loss function correctness, target alignment, and gradient flow
# by fitting 4 samples to near-zero loss without regularization (dropout=0, weight_decay=0).

# %%
import torch
import torch.nn as nn
import torch.optim as optim


In [2]:

# 1. Define a minimal 2-layer network without regularizers
class TwoLayerClassifier(nn.Module):
    def __init__(self, in_features=8, hidden_dim=16, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [3]:

# 2. Fix seed and create a tiny batch (4 samples)
torch.manual_seed(42)
X_batch = torch.randn(4, 8)
y_batch = torch.randint(0, 2, (4,))

model = TwoLayerClassifier(in_features=8, hidden_dim=16, num_classes=2)
criterion = nn.CrossEntropyLoss()

# No weight decay or L2 penalties
optimizer = optim.Adam(model.parameters(), lr=0.05, weight_decay=0.0)

# 3. Training loop targeting single-batch convergence
print("Starting Single-Batch Overfitting Test...")
for step in range(1, 201):
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = criterion(logits, y_batch)
    loss.backward()
    optimizer.step()

    if step % 25 == 0 or step == 1:
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y_batch).float().mean().item() * 100
        print(f"Step {step:03d} | Loss: {loss.item():.6f} | Accuracy: {acc:.1f}%")

# 4. Diagnostic assertion
final_loss = loss.item()
print("\n--- Diagnostic Verdict ---")
if final_loss < 1e-3:
    print(f"PASSED: Loss reached {final_loss:.6f} (< 1e-3). Forward/backward pass is intact.")
else:
    print(f"FAILED: Loss plateaued at {final_loss:.6f}. Check loss formulation or label alignment.")

Starting Single-Batch Overfitting Test...
Step 001 | Loss: 0.630291 | Accuracy: 25.0%
Step 025 | Loss: 0.000002 | Accuracy: 100.0%
Step 050 | Loss: 0.000000 | Accuracy: 100.0%
Step 075 | Loss: 0.000000 | Accuracy: 100.0%
Step 100 | Loss: 0.000000 | Accuracy: 100.0%
Step 125 | Loss: 0.000000 | Accuracy: 100.0%
Step 150 | Loss: 0.000000 | Accuracy: 100.0%
Step 175 | Loss: 0.000000 | Accuracy: 100.0%
Step 200 | Loss: 0.000000 | Accuracy: 100.0%

--- Diagnostic Verdict ---
PASSED: Loss reached 0.000000 (< 1e-3). Forward/backward pass is intact.


# Diagnostic 1b: Injected Loss Formulation Bug (Double Softmax)

**Goal:** show that the single-batch overfitting test catches a loss formulation bug.

`nn.CrossEntropyLoss` expects **raw logits**. It applies `log_softmax` internally. If the model (or the
training loop) already applies `softmax`, the loss sees probabilities in [0, 1] as if they were logits.

With 2 classes, the largest possible logit gap is then 1 (probabilities 1 vs 0), so the best achievable loss is

`log(1 + e^-1) ≈ 0.3133`

The loss can never reach `1e-3`, no matter how long we train. Everything else (data, seed, init, optimizer, steps) is identical to the baseline above.

In [4]:
# BUGGY RUN: identical to the baseline except for ONE line (double softmax)
import math

torch.manual_seed(42)
X_batch = torch.randn(4, 8)
y_batch = torch.randint(0, 2, (4,))

model_bug = TwoLayerClassifier(in_features=8, hidden_dim=16, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_bug.parameters(), lr=0.05, weight_decay=0.0)

print("Starting Single-Batch Overfitting Test (BUGGY: softmax before CrossEntropyLoss)...")
for step in range(1, 201):
    optimizer.zero_grad()
    logits = model_bug(X_batch)
    probs = torch.softmax(logits, dim=1)      # <-- BUG: CrossEntropyLoss applies log_softmax again
    loss = criterion(probs, y_batch)          # <-- loss is computed on probabilities, not logits
    loss.backward()
    optimizer.step()

    if step % 25 == 0 or step == 1:
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y_batch).float().mean().item() * 100
        print(f"Step {step:03d} | Loss: {loss.item():.6f} | Accuracy: {acc:.1f}%")

final_loss = loss.item()
print("\n--- Diagnostic Verdict ---")
if final_loss < 1e-3:
    print(f"PASSED: Loss reached {final_loss:.6f} (< 1e-3). Forward/backward pass is intact.")
else:
    print(f"FAILED: Loss plateaued at {final_loss:.6f}. Check loss formulation or label alignment.")

Starting Single-Batch Overfitting Test (BUGGY: softmax before CrossEntropyLoss)...
Step 001 | Loss: 0.659117 | Accuracy: 25.0%
Step 025 | Loss: 0.313263 | Accuracy: 100.0%
Step 050 | Loss: 0.313262 | Accuracy: 100.0%
Step 075 | Loss: 0.313262 | Accuracy: 100.0%
Step 100 | Loss: 0.313262 | Accuracy: 100.0%
Step 125 | Loss: 0.313262 | Accuracy: 100.0%
Step 150 | Loss: 0.313262 | Accuracy: 100.0%
Step 175 | Loss: 0.313262 | Accuracy: 100.0%
Step 200 | Loss: 0.313262 | Accuracy: 100.0%

--- Diagnostic Verdict ---
FAILED: Loss plateaued at 0.313262. Check loss formulation or label alignment.


In [5]:
# WHY IT FAILS: the loss has a hard floor, even though the model is "perfect"
floor = math.log(1 + math.exp(-1))   # best case for 2 classes when inputs are probabilities
print(f"Theoretical minimum loss with double softmax (2 classes): {floor:.6f}")
print(f"Observed final loss:                                        {final_loss:.6f}")
print(f"Test threshold:                                             {1e-3:.6f}")

with torch.no_grad():
    final_logits = model_bug(X_batch)
    final_probs = torch.softmax(final_logits, dim=1)
print("\nFinal predicted probabilities (rows sum to 1):")
print(final_probs)
print("Labels:", y_batch.tolist())
print("Predictions:", final_logits.argmax(dim=1).tolist())
print(f"Accuracy: {(final_logits.argmax(1) == y_batch).float().mean().item()*100:.1f}%")

Theoretical minimum loss with double softmax (2 classes): 0.313262
Observed final loss:                                        0.313262
Test threshold:                                             0.001000

Final predicted probabilities (rows sum to 1):
tensor([[4.0069e-07, 1.0000e+00],
        [1.6006e-08, 1.0000e+00],
        [3.1453e-07, 1.0000e+00],
        [1.0000e+00, 2.7440e-08]])
Labels: [1, 1, 1, 0]
Predictions: [1, 1, 1, 0]
Accuracy: 100.0%


## Takeaways

- **Accuracy is 100% but the test still fails.** Accuracy only checks the argmax. The loss is what reveals the bug, which is why the pass condition checks loss, not accuracy.
- **The plateau is a signature.** A loss that flattens at a clean constant (about 0.3133 for 2 classes) instead of heading to zero points to a loss/output mismatch, not an optimizer or learning-rate problem.
- **Fix:** pass raw logits to `CrossEntropyLoss` (remove the `softmax`). Apply `softmax` only at inference time when you need probabilities.
- **Related variants with the same symptom:** `LogSoftmax` followed by `CrossEntropyLoss`, or `Sigmoid` followed by `BCEWithLogitsLoss`.